# EDS � Full-sequence stride results
Run all cells to view saved scores. Each dataset mean weights the available EDS test sequences equally. Stride 1 uses all frames, and strides 2-4 retain approximately one half, one third and one quarter. Set REPORT_ID to select a run. The chart also works as a standalone offline HTML file.

In [ ]:
from pathlib import Path
import html
import json
import numpy as np
import pandas as pd
from IPython.display import HTML, FileLink, display

REPORT_ID = "8fa73a5e847f7f56"  # Set to None to use the most recently written CSV.
REPORT_ROOT = None  # Optional path to reports/experiments/eds/test
METRICS = None  # None shows every metric saved in this report.
SEQUENCES = ["02_rocket_earth_light", "06_ziggy_and_fuzz", "10_office", "14_ziggy_in_the_arena"]
STEPS = [1, 2, 3, 4]


In [ ]:
if REPORT_ROOT is None:
    relative = Path("reports/experiments/eds/test")
    roots = [p / relative for p in (Path.cwd(), *Path.cwd().parents) if (p / relative).is_dir()]
    if not roots:
        raise FileNotFoundError("Open this notebook inside the benchmark repo or set REPORT_ROOT.")
    root = roots[0]
else:
    root = Path(REPORT_ROOT).expanduser()

files = sorted(root.glob("full_sequence_strides*/*/per_sequence.csv"),
               key=lambda p: p.stat().st_mtime_ns, reverse=True)
matches = [p for p in files if REPORT_ID is None or p.parent.name == REPORT_ID]
if not matches:
    raise FileNotFoundError(f"No matching report under {root}. Available: {[str(p.relative_to(root)) for p in files]}")
path = matches[0]
print("Reading", path)
df = pd.read_csv(path)
keys = ["metric", "method", "stride", "sequence"]
df = df[df.sequence.isin(SEQUENCES) & df.stride.isin(STEPS)].copy()
metrics = list(dict.fromkeys(df.metric)) if METRICS is None else list(METRICS)
if not metrics or set(metrics) - set(df.metric):
    raise ValueError(f"Choose from the saved metrics: {sorted(df.metric.unique())}")
df = df[df.metric.isin(metrics)].copy()
if df.duplicated(keys).any():
    raise ValueError("Duplicate scores for a metric, method, stride and sequence.")
df["value"] = pd.to_numeric(df.value, errors="coerce")
accepted = df.status.eq("ok") | (df.metric.eq("fano_factor") & df.status.eq("partial_support"))
df["valid"] = df.value.where(accepted & np.isfinite(df.value))
agg = df.groupby(["metric", "method", "stride"], as_index=False).agg(
    mean=("valid", "mean"), n=("valid", "count"))
methods = sorted(df.method.unique(), key=lambda m: (m != "ground_truth", m))
display(pd.crosstab(df.metric, df.status))

## Dataset means by stride
Fano error uses jointly active cells. Its mean includes finite partial_support scores, so methods can have different spatial-temporal coverage. Each cell shows the mean and the number of contributing sequences. A dash indicates no valid score. The three best distinct scores are shaded for error metrics and event count ratio (closest to 1), among models with complete coverage. Descriptive metrics and recorded GT are shown without ranking.

In [ ]:
lower = {"fano_factor", "chamfer", "chamfer_full", "chamfer_penalty", "chamfer_count", "iei_distance"}
for step in STEPS:
    group = agg[agg.stride.eq(step)]
    values = group.pivot(index="method", columns="metric", values="mean").reindex(index=methods, columns=metrics)
    counts = group.pivot(index="method", columns="metric", values="n").reindex_like(values).fillna(0)
    colors = {}
    for metric in metrics:
        if metric not in lower and not metric.startswith(("voxel_", "pmse_")) and metric != "event_count_ratio":
            continue
        eligible = values.loc[(counts[metric] == len(SEQUENCES)) & (values.index != "ground_truth"), metric].dropna()
        errors = (eligible - 1).abs() if metric == "event_count_ratio" else eligible
        for rank, val in enumerate(sorted(errors.unique())[:3]):
            for method in errors.index[errors == val]:
                colors[method, metric] = ["#9dc9ff", "#c4dfff", "#e2efff"][rank]
    rows = []
    for method in methods:
        cells = []
        for metric in metrics:
            value = values.at[method, metric]
            label = "-" if pd.isna(value) else f"{value:.5g} ({int(counts.at[method, metric])}/{len(SEQUENCES)})"
            color = colors.get((method, metric))
            style = f' style="background:{color};color:#102641"' if color else ""
            cells.append(f"<td{style}>{label}</td>")
        rows.append("<tr><th>" + html.escape(method) + "</th>" + "".join(cells) + "</tr>")
    headers = "".join(f"<th>{html.escape(metric)}</th>" for metric in metrics)
    display(HTML(f"<h3>Stride {step} � M={step-1}</h3><div style='overflow-x:auto'><table>"
                 f"<thead><tr><th>Method</th>{headers}</tr></thead><tbody>{''.join(rows)}</tbody></table></div>"))

## Interactive comparison
Choose a metric and either the dataset mean or one sequence. Hover over points for exact scores and coverage. Missing scores leave gaps. The chart includes recorded GT wherever the report supplies it.

In [ ]:
import plotly.graph_objects as go

# Embed each plot with its own controls. One local copy of Plotly serves the entire dashboard.
from plotly.offline import get_plotlyjs

panels = []
options = []
for i, metric in enumerate(metrics):
    fig = go.Figure()
    choices = ["Dataset mean", *SEQUENCES]
    for choice in choices:
        for method in methods:
            if choice == "Dataset mean":
                subset = agg[(agg.metric == metric) & (agg.method == method)].set_index("stride").reindex(STEPS)
                y = subset["mean"]
                info = [f"{int(n)}/{len(SEQUENCES)} sequences" for n in subset.n.fillna(0)]
            else:
                subset = df[(df.metric == metric) & (df.method == method) & (df.sequence == choice)].set_index("stride").reindex(STEPS)
                y = subset.valid
                info = subset.status.fillna("no saved row").tolist()
            fig.add_scatter(x=STEPS, y=y, mode="lines+markers", connectgaps=False,
                            name="Recorded GT" if method == "ground_truth" else method,
                            visible=choice == choices[0], customdata=info,
                            line=dict(dash="dot" if method == "ground_truth" else "solid"),
                            hovertemplate="Stride %{x} � %{y:.6g}<br>%{customdata}<extra>%{fullData.name}</extra>")
    buttons = [dict(label=c, method="update", args=[dict(visible=[
        j // len(methods) == k for j in range(len(fig.data))])]) for k, c in enumerate(choices)]
    fig.update_layout(title=metric, template="plotly_white", height=510, margin=dict(t=115),
                      xaxis=dict(title="Input frame stride", tickvals=STEPS),
                      yaxis_title=metric, updatemenus=[dict(buttons=buttons, x=0, y=1.20)])
    if metric == "event_count_ratio":
        fig.add_hline(y=1, line_dash="dot")
    options.append(f'<option value="{i}">{html.escape(metric)}</option>')
    plot = fig.to_html(full_html=False, include_plotlyjs=False, config={"responsive": True})
    panels.append(f'<section id="p{i}" {"hidden" if i else ""}>{plot}</section>')

dashboard = ("<!doctype html><meta charset=utf-8><style>body{font:16px system-ui;color:#173354;background:white}"
             "select{font:inherit}section[hidden]{display:none}</style><script>" + get_plotlyjs() + "</script>"
             + "<label>Metric <select id='metric'>" + "".join(options) + "</select></label>" + "".join(panels)
             + """<script>document.getElementById('metric').onchange=function(){
                 document.querySelectorAll('section').forEach(s=>s.hidden=s.id!=='p'+this.value);
                 Plotly.Plots.resize(document.querySelector('section:not([hidden]) .js-plotly-plot'));
             };</script>""")
out_dir = Path.cwd() / "stride_figures" / path.parent.parent.name / path.parent.name
out_dir.mkdir(parents=True, exist_ok=True)
out = out_dir / "stride_comparison.html"
out.write_text(dashboard, encoding="utf-8")
display(FileLink(str(out.relative_to(Path.cwd()))))
display(HTML('<iframe title="EDS stride comparison" width="100%" height="600" style="border:0" srcdoc="'
             + html.escape(dashboard, quote=True) + '"></iframe>'))